# Assignment 3 – Parte 1: Visualización

## Pregunta

> **¿Qué regiones del Perú crecieron más entre 2007 y 2019, y tuvo que ver la minería?**

Usamos el **Valor Agregado Bruto (VAB) real por habitante**: cuánto produce un departamento en un año,
en soles de 2007 (sin inflación), dividido entre su población.

In [1]:
import pandas as pd
import plotly.express as px

## 1. Cargar los datos

`dataset.csv` tiene una fila por departamento y año (24 departamentos × 13 años, 2007–2019).
El `ubigeo` se lee como **texto** para no perder el cero de la izquierda (`'01'`, no `1`).

In [2]:
dataset = pd.read_csv("datos/dataset.csv", dtype={"ubigeo": str})
dataset.shape

(312, 9)

In [3]:
dataset.head()

,ubigeo,departamento,anio,region_natural,poblacion,vab_pc,peso_mineria_pct,peso_agricultura_pct,peso_manufactura_pct
0,01,AMAZONAS,2007,Selva,407893,4360.9,2.8,38.0,6.6
1,01,AMAZONAS,2008,Selva,407690,4736.3,2.8,37.8,6.9
2,01,AMAZONAS,2009,Selva,407451,5051.7,2.9,36.3,5.8
3,01,AMAZONAS,2010,Selva,407420,5426.1,3.3,35.3,5.9
4,01,AMAZONAS,2011,Selva,407347,5614.6,3.2,31.8,5.4


## 2. Tema y columnas que usamos

**Tema:** crecimiento económico regional y peso de la minería.

| Columna | Qué es |
|---|---|
| `departamento`, `anio` | identificadores |
| `vab_pc` | VAB real por habitante, en soles de 2007 |
| `peso_mineria_pct` | % del VAB que viene de "extracción de petróleo, gas y minerales" |
| `region_natural` | Costa, Sierra o Selva (INEI) |

**Fuentes:** INEI, Producto Bruto Interno por departamentos (VAB a precios constantes de 2007) y
estimaciones de población al 30 de junio.

**Dos decisiones que conviene saber:**

1. **Lima y Callao van juntos.** El INEI publica el VAB de Lima *con* la producción del Callao
   (refinería, puerto, aeropuerto), pero cuenta la población del Callao por separado. Si dividimos
   el VAB de Lima solo entre la gente de Lima, el VAB por habitante sale ~11 % inflado. Juntándolos,
   la división es consistente.
2. **"Minería" incluye petróleo y gas**, porque así lo agrupa el INEI. Por eso Loreto aparece con
   un peso minero de 20 %.

In [4]:
# Comprobamos que el ubigeo quedó como texto de 2 dígitos
dataset["ubigeo"].str.len().value_counts()

ubigeo
2    312
Name: count, dtype: int64

## 3. Barras: ¿quién creció más?

Comparamos el VAB por habitante de 2007 con el de 2019.

In [5]:
vab_2007 = dataset[dataset["anio"] == 2007][["departamento", "vab_pc"]]
vab_2019 = dataset[dataset["anio"] == 2019][["departamento", "vab_pc"]]

crecimiento = vab_2007.merge(vab_2019, on="departamento", suffixes=("_2007", "_2019"))
crecimiento.head()

,departamento,vab_pc_2007,vab_pc_2019
0,AMAZONAS,4360.9,7476.4
1,ANCASH,14298.2,17151.5
2,APURIMAC,4192.4,16691.6
3,AREQUIPA,14397.1,21441.7
4,AYACUCHO,4612.0,8926.4


In [6]:
crecimiento["crecimiento_pct"] = (crecimiento["vab_pc_2019"] / crecimiento["vab_pc_2007"] - 1) * 100
crecimiento["crecimiento_pct"] = crecimiento["crecimiento_pct"].round(1)

# Una etiqueta de texto para pintar distinto a quienes retrocedieron
crecimiento["tendencia"] = "Creció"
crecimiento.loc[crecimiento["crecimiento_pct"] < 0, "tendencia"] = "Retrocedió"

crecimiento = crecimiento.sort_values("crecimiento_pct", ascending=False)
crecimiento

,departamento,vab_pc_2007,vab_pc_2019,crecimiento_pct,tendencia
2,APURIMAC,4192.4,16691.6,298.1,Creció
8,HUANUCO,4075.0,8003.5,96.4,Creció
4,AYACUCHO,4612.0,8926.4,93.5,Creció
6,CUSCO,8985.0,16417.4,82.7,Creció
7,HUANCAVELICA,5343.0,9502.3,77.8,Creció
19,PUNO,4531.7,7912.5,74.6,Creció
0,AMAZONAS,4360.9,7476.4,71.4,Creció
20,SAN MARTIN,4349.7,6877.3,58.1,Creció
10,JUNIN,7242.3,11355.7,56.8,Creció
12,LAMBAYEQUE,6033.2,9375.1,55.4,Creció


In [7]:
fuente = "Fuente: INEI, VAB por departamentos (soles de 2007) y estimaciones de población. Elaboración propia."

fig_barras = px.bar(
    crecimiento.sort_values("crecimiento_pct"),
    x="crecimiento_pct", y="departamento", orientation="h",
    color="tendencia", text="crecimiento_pct",
    color_discrete_map={"Creció": "#4C78A8", "Retrocedió": "#E45756"},
    labels={"crecimiento_pct": "Crecimiento del VAB por habitante, 2007-2019 (%)",
            "departamento": "", "tendencia": ""},
    title="Apurímac casi cuadruplicó su producción por habitante<br>y solo Madre de Dios y Moquegua retrocedieron, 2007-2019",
)
fig_barras.update_traces(texttemplate="%{text:.0f} %", textposition="outside", cliponaxis=False)
fig_barras.update_xaxes(range=[-60, 340])
fig_barras.update_layout(
    template="plotly_white",
    height=650,
    margin=dict(l=10, r=10, t=90, b=110),
    annotations=[dict(text=fuente, showarrow=False, xref="paper", yref="paper",
                      x=0, y=-0.13, font=dict(size=11, color="gray"))],
)
fig_barras.show()

### Verificación: ¿el gráfico dice lo mismo que la tabla?

Los 3 que más crecieron y los 3 que menos crecieron:

In [8]:
crecimiento.head(3)

,departamento,vab_pc_2007,vab_pc_2019,crecimiento_pct,tendencia
2,APURIMAC,4192.4,16691.6,298.1,Creció
8,HUANUCO,4075.0,8003.5,96.4,Creció
4,AYACUCHO,4612.0,8926.4,93.5,Creció


In [9]:
crecimiento.tail(3)

,departamento,vab_pc_2007,vab_pc_2019,crecimiento_pct,tendencia
17,PASCO,19220.6,20002.7,4.1,Creció
16,MOQUEGUA,45665.3,44350.3,-2.9,Retrocedió
15,MADRE DE DIOS,16398.2,12673.3,-22.7,Retrocedió


Coincide con el gráfico: la barra más larga es Apurímac (+298 %), seguida de lejos por Huánuco y
Ayacucho, y las únicas barras rojas son Madre de Dios (−22.7 %) y Moquegua (−2.9 %).

## 4. Dispersión: ¿más minería, más producción por habitante?

Cada punto es un departamento en 2019. Pasa el mouse sobre un punto para ver cuál es.

In [10]:
datos_2019 = dataset[dataset["anio"] == 2019]

fig_dispersion = px.scatter(
    datos_2019,
    x="peso_mineria_pct", y="vab_pc",
    color="region_natural", hover_name="departamento",
    labels={"peso_mineria_pct": "Peso de la minería en la economía regional, 2019 (%)",
            "vab_pc": "VAB por habitante, 2019 (soles de 2007)",
            "region_natural": "Región natural"},
    title="Los departamentos más mineros suelen producir más por habitante,<br>pero Moquegua escapa al patrón (2019)",
)
fig_dispersion.update_traces(marker=dict(size=11))
fig_dispersion.update_layout(
    template="plotly_white",
    margin=dict(l=10, r=10, t=90, b=110),
    annotations=[dict(text=fuente, showarrow=False, xref="paper", yref="paper",
                      x=0, y=-0.25, font=dict(size=11, color="gray"))],
)
fig_dispersion.show()

### Verificación

¿Quiénes tienen más peso minero y quiénes producen más por habitante?

In [11]:
columnas = ["departamento", "peso_mineria_pct", "vab_pc"]
datos_2019.sort_values("peso_mineria_pct", ascending=False)[columnas].head(3)

,departamento,peso_mineria_pct,vab_pc
38,APURIMAC,61.9,16691.6
233,PASCO,58.0,20002.7
285,TACNA,45.4,24056.6


In [12]:
datos_2019.sort_values("vab_pc", ascending=False)[columnas].head(3)

,departamento,peso_mineria_pct,vab_pc
220,MOQUEGUA,26.6,44350.3
285,TACNA,45.4,24056.6
51,AREQUIPA,34.9,21441.7


In [13]:
# Qué tan juntas se mueven las dos variables (de -1 a 1)
correlacion_todos = datos_2019["peso_mineria_pct"].corr(datos_2019["vab_pc"])

sin_moquegua = datos_2019[datos_2019["departamento"] != "MOQUEGUA"]
correlacion_sin_moquegua = sin_moquegua["peso_mineria_pct"].corr(sin_moquegua["vab_pc"])

print("Con los 24 departamentos:", round(correlacion_todos, 2))
print("Sin Moquegua:", round(correlacion_sin_moquegua, 2))

Con los 24 departamentos: 0.45
Sin Moquegua: 0.64


En el gráfico, los puntos más a la derecha son Apurímac, Pasco y Tacna, y el más alto, muy separado
del resto, es Moquegua. Coincide con la tabla.

Moquegua produce mucho por habitante sin ser de los más mineros, porque su fuerte es la
**manufactura** (44 % de su VAB). Sin Moquegua, la relación entre minería y producción por
habitante pasa de 0.45 a 0.64.

## 5. Línea: ¿cuándo despegó Apurímac?

Seguimos a los dos extremos del gráfico de barras año por año y los comparamos con el
promedio simple de los 24 departamentos.

In [14]:
extremos = dataset[dataset["departamento"].isin(["APURIMAC", "MADRE DE DIOS"])]
extremos = extremos[["anio", "departamento", "vab_pc"]]

promedio = dataset.groupby("anio")["vab_pc"].mean().reset_index()
promedio["departamento"] = "PROMEDIO 24 DEPARTAMENTOS"

evolucion = pd.concat([extremos, promedio])
evolucion.head()

,anio,departamento,vab_pc
26,2007,APURIMAC,4192.4
27,2008,APURIMAC,3898.0
28,2009,APURIMAC,3765.9
29,2010,APURIMAC,4112.3
30,2011,APURIMAC,4372.8


In [15]:
fig_linea = px.line(
    evolucion, x="anio", y="vab_pc", color="departamento", markers=True,
    labels={"anio": "", "vab_pc": "VAB por habitante (soles de 2007)", "departamento": ""},
    title="Apurímac dio el salto en 2016;<br>Madre de Dios fue perdiendo terreno desde 2011",
)
fig_linea.add_vline(x=2016, line_dash="dash", line_color="gray")
fig_linea.update_layout(
    template="plotly_white",
    margin=dict(l=10, r=10, t=90, b=110),
    annotations=[
        dict(text="2016: inicia producción de Las Bambas", showarrow=False,
             x=2016, y=1.02, xref="x", yref="paper", xanchor="left", font=dict(size=11)),
        dict(text=fuente, showarrow=False, xref="paper", yref="paper",
             x=0, y=-0.22, font=dict(size=11, color="gray")),
    ],
)
fig_linea.show()

### Verificación

El salto de Apurímac coincide con el aumento del peso de la minería en su economía:

In [16]:
columnas = ["anio", "vab_pc", "peso_mineria_pct"]
apurimac = dataset[dataset["departamento"] == "APURIMAC"]
apurimac[apurimac["anio"].between(2014, 2017)][columnas]

,anio,vab_pc,peso_mineria_pct
33,2014,5775.7,4.7
34,2015,6239.8,10.4
35,2016,15012.0,60.9
36,2017,18173.4,67.4


In [17]:
madre_de_dios = dataset[dataset["departamento"] == "MADRE DE DIOS"]
madre_de_dios[madre_de_dios["anio"].isin([2007, 2011, 2019])][columnas]

,anio,vab_pc,peso_mineria_pct
195,2007,16398.2,53.0
199,2011,19312.9,53.5
207,2019,12673.3,25.5


Entre 2015 y 2016 el VAB por habitante de Apurímac pasó de 6 240 a 15 012 soles, y la minería pasó
de 10 % a 61 % de su economía. En Madre de Dios ocurre lo contrario: la minería bajó de 53 % (2007)
a 25 % (2019) y su VAB por habitante cayó.

## 6. Guardar un gráfico en HTML

El archivo se abre con doble clic en cualquier navegador y sigue siendo interactivo.
`include_plotlyjs="cdn"` hace que el archivo pese unos KB en vez de 4 MB: la librería de
Plotly se descarga de internet al abrirlo.

In [18]:
fig_barras.write_html("grafico.html", include_plotlyjs="cdn")
print("Guardado: grafico.html")

Guardado: grafico.html


## 7. Lo que encontramos

1. **Apurímac es el caso extremo:** su producción por habitante casi se cuadruplicó (+298 %), y casi
   todo el salto ocurre en 2016, el año en que la minería pasa a ser el 61 % de su economía.
2. **Madre de Dios y Moquegua son los únicos que retrocedieron.** En Madre de Dios, la caída va de la
   mano con la pérdida de peso de la minería.
3. **Más minería suele ir con más producción por habitante**, pero es una asociación, no una causa.
   Moquegua muestra que la manufactura también puede llevar a un VAB por habitante alto.

**Limitaciones:**
- El VAB mide producción dentro del departamento, no ingresos de su población: parte de las
  ganancias mineras sale de la región.
- Las correlaciones no prueban que la minería *cause* el crecimiento.
- Lima y Callao aparecen juntos porque el INEI no publica el VAB del Callao por separado.

---
## Qué hicimos

| Paso | Código |
|---|---|
| Leer el ubigeo como texto | `pd.read_csv(..., dtype={"ubigeo": str})` |
| Comparar dos años | `.merge(..., on="departamento", suffixes=(...))` |
| Calcular crecimiento | `(vab_2019 / vab_2007 - 1) * 100` |
| Barras horizontales | `px.bar(..., orientation="h", color="tendencia")` |
| Dispersión con nombre al pasar el mouse | `px.scatter(..., hover_name="departamento")` |
| Medir relación entre dos variables | `serie_a.corr(serie_b)` |
| Evolución en el tiempo | `px.line(..., color="departamento", markers=True)` |
| Marcar un año clave | `fig.add_vline(x=2016, ...)` |
| Título, ejes y fuente | `title=`, `labels=`, `annotations=[...]` |
| Guardar interactivo | `fig.write_html("grafico.html")` |